# Web Search Tool

Similar to Text Editor, but way easier **(you don't have to code anyting, it is Claude the one that performs the web search)**

> 👁️👁️ We can tweak to filter and narrow down results from web search (e.g. only include official sources, pubmed sources, etc.)


In [3]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-sonnet-4-5"

In [15]:
# 👁️👁️ We can tweak to filter and narrow down results from web search (e.g. only include official sources, pubmed sources, etc.)

web_search_schema = {
    "type":"web_search_20250305",
    "name":"web_search",
    "max_uses": 5, # so that Claude do not get stuck in consecutive searches
    "allowed_domains": ["nih.gov"] # IMPORTANTE (p.ej podemos usar morningstar como fuente si la app va de fondos)
}

In [16]:
# Helper functions
from anthropic.types import Message


def add_user_message(messages, message):
    user_message = {
        "role": "user",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message,
    }
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[], tools=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join([block.text for block in message.content if block.type == "text"])

In [17]:
messages = []
add_user_message(
    messages,
    """
    What's the best exercise for gaining leg muscle? Do some research online
    """,
)
response = chat(messages,tools=[web_search_schema])
response # Quite large response, it contains several blocks that we've not seen before (contiene las urls, las fuentes, contenido, etc.)
# TODO ESTO SE PUEDE CAPTURAR EN LA RESPUESTA Y PONERLO EN NUESTRA APP

Message(id='msg_01SPAhbKkFCgyq2ooEvtKeCS', container=None, content=[ServerToolUseBlock(id='srvtoolu_01S5YTcHLBokRrz6DoB8PTH6', caller=None, input={'query': 'best exercise gaining leg muscle'}, name='web_search', type='server_tool_use'), WebSearchToolResultBlock(caller=DirectCaller(type='direct'), content=[WebSearchResultBlock(encrypted_content='EtwhCioIDhgCIiRjYjNiNDAyNC1mYjg5LTQxYTctYjg4Ni1lMWM0MjBlMmJhNjISDBkm1bZUCkOD48aEGBoMByRSoTgBnWHtZqP5IjBiQS3crpYsDvJctosIzNkQS6w3o/TSyRbM7opAw4QbHJHC82/vNgGGn10oub2UEo8q3yBTgSNXUOUAtr4Keyl+stY7DrJlcmBPbsBEn5MguaPP/Hm9kf7Wey0uoyYijioj5zkrviGUebyo1/8HmyLQe0OxKWMOltiQQj4nrRpTED5iXIGyC/ihT/A2vypkU86It3M4FI7Gn8LFW0pHky6Fl6UQPNpVTX7JBPzmCvostUmlr8oq0eB8Zs/SIUC3k133VzTwMCkxpYsrZPPoClQmbXzeO/eN8ioN2C9wLeByUrxvOb7gdwQ5+yMxNB0tQAAeagiBTxlCsLyA8+M/shjB0j+JtTZmH+iUhsJfO3yiqQMV4SXicsnNtgwA3bRsBS3S9iV+7Fr2hZvrtOlT3KPY74259TFZJwchPuZa0tB1S7Ou8KdgFJNTHzkdaVNNWXUXmfrwg40MYR1GbgSxvA2Rvy9Tp7VTlSwesqFpZkOpcrsjnDItni/dF8op2mRynQHgA93knx2E1g5A8jP+GxefY4I7arP2VQbjfkYkk

In [18]:
for block in response.content:
    print(block)

ServerToolUseBlock(id='srvtoolu_01S5YTcHLBokRrz6DoB8PTH6', caller=None, input={'query': 'best exercise gaining leg muscle'}, name='web_search', type='server_tool_use')
WebSearchToolResultBlock(caller=DirectCaller(type='direct'), content=[WebSearchResultBlock(encrypted_content='EtwhCioIDhgCIiRjYjNiNDAyNC1mYjg5LTQxYTctYjg4Ni1lMWM0MjBlMmJhNjISDBkm1bZUCkOD48aEGBoMByRSoTgBnWHtZqP5IjBiQS3crpYsDvJctosIzNkQS6w3o/TSyRbM7opAw4QbHJHC82/vNgGGn10oub2UEo8q3yBTgSNXUOUAtr4Keyl+stY7DrJlcmBPbsBEn5MguaPP/Hm9kf7Wey0uoyYijioj5zkrviGUebyo1/8HmyLQe0OxKWMOltiQQj4nrRpTED5iXIGyC/ihT/A2vypkU86It3M4FI7Gn8LFW0pHky6Fl6UQPNpVTX7JBPzmCvostUmlr8oq0eB8Zs/SIUC3k133VzTwMCkxpYsrZPPoClQmbXzeO/eN8ioN2C9wLeByUrxvOb7gdwQ5+yMxNB0tQAAeagiBTxlCsLyA8+M/shjB0j+JtTZmH+iUhsJfO3yiqQMV4SXicsnNtgwA3bRsBS3S9iV+7Fr2hZvrtOlT3KPY74259TFZJwchPuZa0tB1S7Ou8KdgFJNTHzkdaVNNWXUXmfrwg40MYR1GbgSxvA2Rvy9Tp7VTlSwesqFpZkOpcrsjnDItni/dF8op2mRynQHgA93knx2E1g5A8jP+GxefY4I7arP2VQbjfkYkketl5HqWC0/t+UfDoC7vbkxMERaaPe1iMu7ytJyHsF8UuxPRVOxipKpLpnSvmJXsyQqXKQ